# H0 確認：Colab（無料版・CPU）で Genesis と SO-101 が動くか

このノートは教材の前の **確認だけ** です。上から順に「▶」を押し、最後のセルの表をそのまま Claude に貼ってください。

- ランタイムは **CPU のまま** でよい（「ランタイム」→「ランタイムのタイプを変更」→ CPU）。無料版の GPU は割り当てられないことがあるので、教材は CPU で作る
- 全部で 10 分ほど（入れるのに数分、形を細かく分ける計算に 1 分ほど）

## 1. 入れる

In [ ]:
import time, os, sys, subprocess
T = {}
t0 = time.time()
!apt-get -qq install -y libosmesa6 > /dev/null
!pip -q install genesis-world==1.4.3 rerun-sdk[notebook]==0.38.1
T["install_s"] = round(time.time() - t0)
print("install", T["install_s"], "s")

## 2. SO-101 の形を取ってくる（TheRobotStudio/SO-ARM100、手元と同じ版 a758567、Apache-2.0）

In [ ]:
if not os.path.exists("SO-ARM100"):
    !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git
!cd SO-ARM100 && git checkout -q a758567 && git log --oneline -1
MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
print(os.path.exists(MODEL), MODEL)

## 3. 2 台の SO-101 と箱を置いて組み立てる

In [ ]:
os.environ["PYOPENGL_PLATFORM"] = "osmesa"   # 画面のない計算機で絵を描く（GPU なし）
import numpy as np
import genesis as gs
gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10),
              rigid_options=gs.options.RigidOptions(box_box_detection=True), show_viewer=False)
sc.add_entity(gs.morphs.Plane())
D = 0.44
armA = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(0, 0, 0), decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
armB = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(D, 0, 0), euler=(0, 0, 180), decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
box = sc.add_entity(gs.morphs.Box(size=(0.03, 0.03, 0.03), pos=(0.2, -0.1, 0.015)), surface=gs.surfaces.Default(color=(0.72, 0.55, 0.36)))
cam = sc.add_camera(res=(768, 432), pos=(0.22, -0.55, 0.30), lookat=(0.22, 0.0, 0.05), fov=55, GUI=False)
t0 = time.time()
sc.build()
T["build_s"] = round(time.time() - t0, 1)
# 関節が全部 0 の姿勢では、前腕を水平に伸ばす（手先は根元から 22 cm 以上）。44 cm 離して向かい合わせると、真ん中で
# 2 台の手先が重なって刺さる。組み立てた直後に、たたんだ姿勢にしてから動かす（作品の共通部品 so101.py の FOLD と同じ）
FOLD = np.array([0.0, -1.7, 1.6, 1.2, 0.0, 0.0])
for a in (armA, armB):
    a.set_dofs_position(FOLD)
    a.control_dofs_position(FOLD)
print("build", T["build_s"], "s   dofs per arm:", armA.n_dofs)

## 4. 速さを測る（1 秒ぶん＝100 step）と、1 コマ描く

In [ ]:
t0 = time.time()
for _ in range(100):
    sc.step()
T["sec_per_sim_sec"] = round(time.time() - t0, 2)
t0 = time.time()
rgb = cam.render(rgb=True)[0]
T["render_s"] = round(time.time() - t0, 2)
T["render_mean"] = round(float(np.asarray(rgb).mean()), 1)   # 0 なら真っ黒（描けていない）
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4.5)); plt.imshow(rgb); plt.axis("off"); plt.show()
print(T)

## 5. 腕を動かして 3 秒の動画にし、ノートの中で再生する

In [ ]:
from IPython.display import HTML
import base64, imageio
frames = []
target = np.array([0.0, -0.6, 0.6, 1.2, 0.0, 0.8])   # たたんだ姿勢から、肩と肘を起こし、指を開く
t0 = time.time()
for k in range(300):
    s = min(1.0, k / 150)
    armA.control_dofs_position(FOLD + (target - FOLD) * s)
    sc.step()
    if k % 4 == 0:
        frames.append(cam.render(rgb=True)[0])
T["video_s"] = round(time.time() - t0, 1)
imageio.mimsave("check.mp4", frames, fps=25)
HTML(f'<video controls width=768 src="data:video/mp4;base64,{base64.b64encode(open("check.mp4","rb").read()).decode()}">')

## 6. rerun をノートの中に出す（講義の第 5 回の道具）

In [ ]:
import rerun as rr
rr.init("H0_check")
for k in range(100):
    rr.set_time("t", duration=k * 0.01)
    rr.log("shoulder_q", rr.Scalars(float(np.sin(k / 10))))
try:
    rr.notebook_show(width=640, height=300)
    T["rerun_notebook"] = "shown"
except Exception as e:
    T["rerun_notebook"] = f"error: {e}"[:80]

## 7. 結果（この表をそのまま Claude に貼る）

In [ ]:
import platform, torch
T["python"] = platform.python_version()
T["torch"] = torch.__version__
T["cpus"] = os.cpu_count()
T["genesis"] = gs.__version__
T["video_ok"] = os.path.getsize("check.mp4") > 10000
for k, v in T.items():
    print(f"{k:16s} {v}")